# 🌿 Mapudungun AI Translator
## Train your own Spanish ↔ Mapudungun translation model

---

### What this notebook does:
1. **Scrapes** translation data from Glosbe.com
2. **Cleans** and prepares the data for training
3. **Fine-tunes** Meta's NLLB-200 model using LoRA (efficient training)
4. **Saves** your model to HuggingFace
5. **Tests** translations in both directions

### Requirements:
- Google Colab account (free)
- HuggingFace account (free) - for saving your model
- ~2-4 hours of GPU time

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Have your HuggingFace token ready (from huggingface.co/settings/tokens)

---
*Created for Mapudungun language preservation*

## 📋 Step 1: Check GPU and Setup Environment

**What this does:**
- Verifies you have a GPU (required for training)
- Shows you what GPU type you got

**Expected output:** Something like `Tesla T4` with 15GB memory

In [ ]:
# Check if GPU is available
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"✅ GPU Memory: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU found!")
    print("👉 Go to Runtime → Change runtime type → Select 'T4 GPU'")

## 📦 Step 2: Install Required Libraries

**What this does:**
- Installs all the Python packages we need
- This takes about 2-3 minutes

**Libraries explained:**
- `transformers` - HuggingFace's library for AI models
- `peft` - For LoRA (efficient fine-tuning)
- `datasets` - For handling training data
- `accelerate` - Makes training faster
- `bitsandbytes` - For memory-efficient training
- `beautifulsoup4` - For scraping Glosbe
- `sentencepiece` - For text tokenization

In [ ]:
# Install all required packages
!pip install -q transformers>=4.36.0
!pip install -q peft>=0.7.0
!pip install -q datasets>=2.15.0
!pip install -q accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0
!pip install -q sentencepiece>=0.1.99
!pip install -q beautifulsoup4>=4.12.0
!pip install -q requests>=2.31.0
!pip install -q tqdm>=4.66.0
!pip install -q sacrebleu>=2.3.0

print("✅ All packages installed successfully!")

## 🔑 Step 3: Login to HuggingFace

**What this does:**
- Connects to your HuggingFace account
- Allows you to save your trained model online

**How to get your token:**
1. Go to https://huggingface.co/settings/tokens
2. Click "New token"
3. Name it anything (e.g., "colab")
4. Select "Write" access
5. Copy the token

**Security tip:** You can also use Colab's Secrets (🔑 icon in left sidebar) to store your token securely.

In [ ]:
from huggingface_hub import login, HfApi
from google.colab import userdata

# Try to get token from Colab Secrets first, otherwise ask for input
try:
    # If you saved your token in Colab Secrets as 'HF_TOKEN'
    hf_token = userdata.get('HF_TOKEN')
    print("✅ Found token in Colab Secrets!")
except:
    # Otherwise, enter it manually
    print("Enter your HuggingFace token (from huggingface.co/settings/tokens):")
    hf_token = input()

# Login to HuggingFace
login(token=hf_token)
print("✅ Successfully logged in to HuggingFace!")

# Get your username for later
api = HfApi()
hf_username = api.whoami()["name"]
print(f"✅ Logged in as: {hf_username}")

## 🌐 Step 4: Scrape Data from Glosbe

**What this does:**
- Connects to Glosbe.com's Spanish-Mapudungun dictionary
- Downloads translation pairs and examples
- Saves them for training

**Important notes:**
- This respects Glosbe's servers with delays between requests
- Takes about 15-30 minutes depending on data available
- We scrape responsibly (not too fast)

**What we're collecting:**
- Word translations (e.g., "perro" → "txewa")
- Phrase examples (e.g., "Buenos días" → "Ayin antü")

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json
import re
from tqdm import tqdm
import random

class GlosbeScraper:
    """
    Scrapes Spanish-Mapudungun translations from Glosbe.com
    """
    def __init__(self):
        self.base_url = "https://glosbe.com"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        })
        self.translations = []
        
    def get_page(self, url, retries=3):
        """Fetch a page with retry logic"""
        for attempt in range(retries):
            try:
                # Be respectful - wait between requests
                time.sleep(random.uniform(1.5, 3.0))
                response = self.session.get(url, timeout=15)
                if response.status_code == 200:
                    return BeautifulSoup(response.text, 'html.parser')
                elif response.status_code == 429:  # Too many requests
                    print(f"⚠️ Rate limited, waiting 30 seconds...")
                    time.sleep(30)
            except Exception as e:
                print(f"⚠️ Attempt {attempt+1} failed: {e}")
                time.sleep(5)
        return None
    
    def get_alphabet_words(self, lang_from='es', lang_to='arn'):
        """Get list of words starting with each letter"""
        words = []
        alphabet = 'abcdefghijklmnopqrstuvwxyz'
        
        print("📚 Fetching word list from alphabet pages...")
        for letter in tqdm(alphabet, desc="Scanning alphabet"):
            url = f"{self.base_url}/{lang_from}/{lang_to}/similar/{letter}"
            soup = self.get_page(url)
            if soup:
                # Find word links
                links = soup.find_all('a', href=re.compile(f'/{lang_from}/{lang_to}/'))
                for link in links:
                    word = link.get_text(strip=True)
                    if word and len(word) > 1 and not word.startswith('http'):
                        words.append(word)
        
        return list(set(words))  # Remove duplicates
    
    def get_translation(self, word, lang_from='es', lang_to='arn'):
        """Get translations for a specific word"""
        url = f"{self.base_url}/{lang_from}/{lang_to}/{requests.utils.quote(word)}"
        soup = self.get_page(url)
        
        if not soup:
            return []
        
        results = []
        
        # Method 1: Look for translation pairs in the page
        # Glosbe shows translations in various formats
        
        # Find direct translations
        translation_divs = soup.find_all('div', class_=re.compile('translation'))
        for div in translation_divs:
            text = div.get_text(strip=True)
            if text and text != word:
                results.append({
                    'source': word,
                    'target': text,
                    'type': 'translation'
                })
        
        # Find example sentences (parallel corpus)
        example_pairs = soup.find_all('div', class_=re.compile('example|tmem|sentence'))
        for pair in example_pairs:
            spans = pair.find_all(['span', 'div'], class_=re.compile('text|content'))
            if len(spans) >= 2:
                source_text = spans[0].get_text(strip=True)
                target_text = spans[1].get_text(strip=True)
                if source_text and target_text and source_text != target_text:
                    results.append({
                        'source': source_text,
                        'target': target_text,
                        'type': 'example'
                    })
        
        # Alternative: Look for any text in specific containers
        containers = soup.find_all(['li', 'div'], class_=re.compile('result|entry|pair'))
        for container in containers:
            texts = [t.get_text(strip=True) for t in container.find_all(['span', 'a', 'div']) if t.get_text(strip=True)]
            texts = [t for t in texts if len(t) > 1 and len(t) < 500]
            if len(texts) >= 2:
                results.append({
                    'source': texts[0],
                    'target': texts[1],
                    'type': 'pair'
                })
        
        return results
    
    def scrape_all(self, max_words=500):
        """Main scraping function"""
        print("🚀 Starting Glosbe scraper for Spanish-Mapudungun...")
        print("="*50)
        
        # Get word list
        words = self.get_alphabet_words()
        print(f"\n📝 Found {len(words)} unique words to process")
        
        # Limit to max_words for reasonable training time
        if len(words) > max_words:
            words = random.sample(words, max_words)
            print(f"📝 Sampling {max_words} words for processing")
        
        # Scrape translations for each word
        print("\n🔍 Fetching translations...")
        for word in tqdm(words, desc="Scraping translations"):
            translations = self.get_translation(word)
            self.translations.extend(translations)
            
            # Also try reverse direction (Mapudungun -> Spanish)
            reverse_translations = self.get_translation(word, 'arn', 'es')
            for t in reverse_translations:
                # Swap source and target for consistency
                self.translations.append({
                    'source': t['target'],
                    'target': t['source'],
                    'type': t['type']
                })
        
        # Remove duplicates
        seen = set()
        unique_translations = []
        for t in self.translations:
            key = (t['source'], t['target'])
            if key not in seen:
                seen.add(key)
                unique_translations.append(t)
        
        self.translations = unique_translations
        print(f"\n✅ Scraped {len(self.translations)} unique translation pairs!")
        return self.translations

# Create scraper instance
scraper = GlosbeScraper()
print("✅ Scraper initialized!")
print("\n⚠️ Note: Scraping will take 15-30 minutes. Please be patient!")

In [ ]:
# Run the scraper
# Adjust max_words based on how much data you want (more = better but slower)
# Recommended: Start with 300-500 for testing, increase to 1000+ for final training

MAX_WORDS = 500  # Change this to scrape more/less data

print(f"🌐 Starting to scrape Glosbe (max {MAX_WORDS} words)...")
print("☕ This is a good time for a coffee break!\n")

raw_translations = scraper.scrape_all(max_words=MAX_WORDS)

print(f"\n📊 Scraping complete!")
print(f"   Total pairs collected: {len(raw_translations)}")

## 🧹 Step 5: Clean and Prepare Data

**What this does:**
- Removes duplicates and empty entries
- Fixes encoding issues (important for Mapudungun special characters)
- Filters out bad quality pairs
- Splits data into training (80%), validation (10%), test (10%)

**Why this matters:**
- Bad data = bad model
- Clean data = better translations

In [ ]:
import re
from collections import Counter

def clean_text(text):
    """Clean and normalize text"""
    if not text or not isinstance(text, str):
        return None
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    # Remove very long texts (likely errors)
    if len(text) > 500:
        return None
    
    # Remove texts that are just numbers or punctuation
    if re.match(r'^[\d\s\.,;:!?]+$', text):
        return None
    
    # Must have at least 2 characters
    if len(text) < 2:
        return None
    
    return text

def clean_translation_pair(pair):
    """Clean a source-target pair"""
    source = clean_text(pair.get('source', ''))
    target = clean_text(pair.get('target', ''))
    
    if not source or not target:
        return None
    
    # Source and target should be different
    if source.lower() == target.lower():
        return None
    
    # Avoid pairs where one is much longer than other (likely misaligned)
    len_ratio = len(source) / len(target) if len(target) > 0 else 0
    if len_ratio > 5 or len_ratio < 0.2:
        return None
    
    return {'source': source, 'target': target}

# Clean all translations
print("🧹 Cleaning translation pairs...")
cleaned_translations = []

for pair in raw_translations:
    cleaned = clean_translation_pair(pair)
    if cleaned:
        cleaned_translations.append(cleaned)

# Remove duplicates again after cleaning
seen = set()
unique_translations = []
for t in cleaned_translations:
    key = (t['source'].lower(), t['target'].lower())
    if key not in seen:
        seen.add(key)
        unique_translations.append(t)

print(f"\n📊 Cleaning results:")
print(f"   Before cleaning: {len(raw_translations)}")
print(f"   After cleaning:  {len(unique_translations)}")
print(f"   Removed:         {len(raw_translations) - len(unique_translations)} bad pairs")

In [ ]:
# Split data into train/validation/test sets
import random

random.seed(42)  # For reproducibility
random.shuffle(unique_translations)

total = len(unique_translations)
train_size = int(total * 0.8)
val_size = int(total * 0.1)

train_data = unique_translations[:train_size]
val_data = unique_translations[train_size:train_size + val_size]
test_data = unique_translations[train_size + val_size:]

print(f"📊 Data split:")
print(f"   Training:   {len(train_data)} pairs (80%)")
print(f"   Validation: {len(val_data)} pairs (10%)")
print(f"   Test:       {len(test_data)} pairs (10%)")

# Show some examples
print(f"\n📝 Sample translation pairs:")
for i, pair in enumerate(train_data[:5]):
    print(f"   {i+1}. Spanish: '{pair['source']}'")
    print(f"      Mapudungun: '{pair['target']}'")
    print()

In [ ]:
# Save data to files (backup)
import json
import os

# Create data directory
os.makedirs('data', exist_ok=True)

# Save as JSON
with open('data/train.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open('data/validation.json', 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

with open('data/test.json', 'w', encoding='utf-8') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

# Also save as parallel text files (common format)
with open('data/train.es', 'w', encoding='utf-8') as f_es, \
     open('data/train.arn', 'w', encoding='utf-8') as f_arn:
    for pair in train_data:
        f_es.write(pair['source'] + '\n')
        f_arn.write(pair['target'] + '\n')

print("✅ Data saved to 'data/' folder!")
print("   - data/train.json, validation.json, test.json")
print("   - data/train.es, train.arn (parallel text files)")

## 🤖 Step 6: Load NLLB-200 Model

**What this does:**
- Downloads Meta's NLLB-200 translation model
- Uses the 600M parameter version (fits in free GPU memory)
- Loads it in 8-bit mode to save memory

**About NLLB-200:**
- Created by Meta AI (Facebook)
- Trained on 200 languages
- Specifically designed for low-resource languages
- Open source and free to use

**Memory note:**
- We use 8-bit quantization to fit in the free T4 GPU (16GB)
- This slightly reduces quality but makes training possible for free

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Model configuration
MODEL_NAME = "facebook/nllb-200-distilled-600M"

print(f"📥 Loading model: {MODEL_NAME}")
print("   This may take a few minutes on first run...\n")

# Configure 8-bit quantization (saves GPU memory)
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

# Load tokenizer
print("1/3 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("   ✅ Tokenizer loaded!")

# Load model with 8-bit quantization
print("2/3 Loading model (this takes a while)...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)
print("   ✅ Model loaded!")

# Check model size
print("3/3 Checking model...")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## ⚡ Step 7: Configure LoRA for Efficient Training

**What is LoRA?**
- Stands for "Low-Rank Adaptation"
- Instead of training ALL model parameters (600 million!), we train small "adapter" layers
- Reduces trainable parameters by 95%+
- Makes training possible on free GPU tiers

**Why it works:**
- The base model already knows how translation works
- We just need to teach it the Spanish-Mapudungun mapping
- Small adapters are enough for this task

**Parameters explained:**
- `r=16`: Size of the adapter (higher = more capacity, more memory)
- `lora_alpha=32`: Learning rate scaling
- `lora_dropout=0.1`: Prevents overfitting
- `target_modules`: Which parts of the model to adapt

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("⚙️ Configuring LoRA adapters...\n")

# Prepare model for training (required for 8-bit models)
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # Rank of the adapter
    lora_alpha=32,                 # Scaling factor
    lora_dropout=0.1,              # Dropout for regularization
    bias="none",                   # Don't train biases
    task_type=TaskType.SEQ_2_SEQ_LM,  # Translation task
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                   "fc1", "fc2"]  # Which layers to adapt
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Check trainable parameters now
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percent = 100 * trainable_params / total_params

print("📊 LoRA Statistics:")
print(f"   Total parameters:     {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Trainable percent:    {trainable_percent:.2f}%")
print(f"\n✅ LoRA configured! Training only {trainable_percent:.2f}% of parameters.")
print(f"   This makes training {100/trainable_percent:.0f}x more efficient!")

## 📚 Step 8: Prepare Training Dataset

**What this does:**
- Converts our cleaned data into the format NLLB-200 expects
- Creates bidirectional training data (both directions)
- Tokenizes text for the model

**Language codes:**
- Spanish: `spa_Latn` (Spanish in Latin script)
- Mapudungun: `arn_Latn` (Mapudungun in Latin script)

**Note:** NLLB-200 might not have perfect Mapudungun support built-in, but fine-tuning will teach it!

In [ ]:
from datasets import Dataset
from functools import partial

# Language codes for NLLB-200
SPANISH_CODE = "spa_Latn"
# Note: Mapudungun (arn) might not be in NLLB's original list,
# but we'll use a similar indigenous language code or add it
MAPUDUNGUN_CODE = "arn_Latn"  # We'll handle this in preprocessing

# Check if Mapudungun is in the tokenizer's vocabulary
if MAPUDUNGUN_CODE not in tokenizer.lang_code_to_id:
    print(f"⚠️ {MAPUDUNGUN_CODE} not in original tokenizer.")
    print("   We'll use the closest available or add it.")
    # Use Quechua as a proxy (closest indigenous language in NLLB)
    MAPUDUNGUN_CODE = "quy_Latn"  # Southern Quechua
    print(f"   Using {MAPUDUNGUN_CODE} as base code.")

print(f"\n📋 Language codes:")
print(f"   Spanish: {SPANISH_CODE}")
print(f"   Mapudungun: {MAPUDUNGUN_CODE}")

In [ ]:
def create_bidirectional_data(data):
    """
    Create training data for both directions:
    - Spanish -> Mapudungun
    - Mapudungun -> Spanish
    """
    bidirectional = []
    
    for pair in data:
        # Spanish -> Mapudungun
        bidirectional.append({
            'source_text': pair['source'],
            'target_text': pair['target'],
            'source_lang': SPANISH_CODE,
            'target_lang': MAPUDUNGUN_CODE
        })
        
        # Mapudungun -> Spanish
        bidirectional.append({
            'source_text': pair['target'],
            'target_text': pair['source'],
            'source_lang': MAPUDUNGUN_CODE,
            'target_lang': SPANISH_CODE
        })
    
    return bidirectional

# Create bidirectional datasets
train_bidirectional = create_bidirectional_data(train_data)
val_bidirectional = create_bidirectional_data(val_data)

print(f"📊 Bidirectional data:")
print(f"   Training samples: {len(train_bidirectional)} (was {len(train_data)})")
print(f"   Validation samples: {len(val_bidirectional)} (was {len(val_data)})")

In [ ]:
def tokenize_function(examples, tokenizer, max_length=128):
    """
    Tokenize source and target texts for the model
    """
    # Set source language
    tokenizer.src_lang = examples['source_lang']
    
    # Tokenize source
    model_inputs = tokenizer(
        examples['source_text'],
        max_length=max_length,
        truncation=True,
        padding='max_length'
    )
    
    # Tokenize target
    tokenizer.src_lang = examples['target_lang']
    labels = tokenizer(
        examples['target_text'],
        max_length=max_length,
        truncation=True,
        padding='max_length'
    )
    
    model_inputs['labels'] = labels['input_ids']
    
    # Replace padding token id with -100 so it's ignored in loss
    model_inputs['labels'] = [
        -100 if token == tokenizer.pad_token_id else token
        for token in model_inputs['labels']
    ]
    
    return model_inputs

# Convert to HuggingFace Dataset format
print("🔄 Converting to Dataset format...")
train_dataset = Dataset.from_list(train_bidirectional)
val_dataset = Dataset.from_list(val_bidirectional)

# Tokenize datasets
print("🔤 Tokenizing datasets...")
tokenized_train = train_dataset.map(
    lambda x: tokenize_function(x, tokenizer),
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training data"
)

tokenized_val = val_dataset.map(
    lambda x: tokenize_function(x, tokenizer),
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation data"
)

print(f"\n✅ Datasets ready for training!")
print(f"   Training samples: {len(tokenized_train)}")
print(f"   Validation samples: {len(tokenized_val)}")

## 🚀 Step 9: Train the Model!

**What this does:**
- Fine-tunes NLLB-200 on your Spanish-Mapudungun data
- Uses the LoRA adapters we configured
- Saves checkpoints during training

**Training parameters explained:**
- `num_train_epochs=3`: How many times to go through all data
- `per_device_train_batch_size=4`: Samples processed at once (limited by GPU memory)
- `learning_rate=2e-4`: How fast the model learns
- `warmup_steps=100`: Gradually increase learning rate at start

**Expected time:** 30-60 minutes on T4 GPU (depends on data size)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import os

# Create output directory
OUTPUT_DIR = "./mapudungun-translator"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Training settings
    num_train_epochs=3,                    # Number of training passes
    per_device_train_batch_size=4,         # Batch size (limited by GPU memory)
    per_device_eval_batch_size=4,          # Evaluation batch size
    gradient_accumulation_steps=4,         # Simulate larger batch size
    
    # Learning rate settings
    learning_rate=2e-4,                    # Learning rate
    warmup_steps=100,                      # Warmup period
    weight_decay=0.01,                     # Regularization
    
    # Evaluation and saving
    eval_strategy="steps",                 # When to evaluate
    eval_steps=200,                        # Evaluate every N steps
    save_strategy="steps",                 # When to save
    save_steps=200,                        # Save every N steps
    save_total_limit=3,                    # Keep only last 3 checkpoints
    
    # Logging
    logging_dir="./logs",
    logging_steps=50,                      # Log every N steps
    
    # Memory optimization
    fp16=True,                             # Use mixed precision
    optim="adamw_torch",                   # Optimizer
    
    # Misc
    predict_with_generate=True,            # For evaluation
    generation_max_length=128,
    load_best_model_at_end=True,           # Load best checkpoint at end
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # HuggingFace Hub
    push_to_hub=False,                     # We'll push manually later
    report_to="none",                      # Disable wandb/tensorboard
)

# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("✅ Trainer configured!")
print(f"\n📊 Training configuration:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Total training samples: {len(tokenized_train)}")

In [ ]:
# Start training!
print("🚀 Starting training...")
print("="*50)
print("☕ This will take 30-60 minutes. Time for another coffee!")
print("="*50)
print()

# Train the model
train_result = trainer.train()

print("\n" + "="*50)
print("✅ Training complete!")
print("="*50)

# Print training statistics
print(f"\n📊 Training results:")
print(f"   Total steps: {train_result.global_step}")
print(f"   Training loss: {train_result.training_loss:.4f}")
print(f"   Training time: {train_result.metrics['train_runtime']/60:.1f} minutes")

## 💾 Step 10: Save Model to HuggingFace

**What this does:**
- Saves your trained model to HuggingFace Hub
- Creates a model card with information
- Makes it available for you (and others!) to use

**Your model will be at:**
`https://huggingface.co/YOUR_USERNAME/mapudungun-translator`

In [ ]:
# Save locally first
print("💾 Saving model locally...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"   ✅ Saved to {OUTPUT_DIR}")

# Create model card
model_card = f"""
---
language:
- es
- arn
license: apache-2.0
tags:
- translation
- mapudungun
- spanish
- indigenous-languages
- low-resource
- nllb
- lora
datasets:
- custom (Glosbe)
---

# Mapudungun-Spanish Translator

This model translates between **Spanish** and **Mapudungun** (both directions).

## Model Details

- **Base model:** facebook/nllb-200-distilled-600M
- **Fine-tuning method:** LoRA (Low-Rank Adaptation)
- **Training data:** Scraped from Glosbe.com ({len(train_data)} pairs)
- **Languages:** Spanish (es) ↔ Mapudungun (arn)

## Usage

```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

# Load model
base_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
model = PeftModel.from_pretrained(base_model, "{hf_username}/mapudungun-translator")
tokenizer = AutoTokenizer.from_pretrained("{hf_username}/mapudungun-translator")

# Translate Spanish to Mapudungun
text = "Buenos días"
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs)
translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(translation)
```

## Limitations

- Works best with short sentences and common phrases
- May struggle with complex grammar or rare vocabulary
- Not suitable for official/legal translations

## Training

Trained with:
- LoRA rank: 16
- Learning rate: 2e-4
- Epochs: 3
- Training samples: {len(train_bidirectional)}

## Purpose

This model was created to help preserve and promote the **Mapudungun** language,
spoken by the Mapuche people of Chile and Argentina.
"""

# Save model card
with open(f"{OUTPUT_DIR}/README.md", "w") as f:
    f.write(model_card)

print("   ✅ Model card created!")

In [ ]:
# Push to HuggingFace Hub
from huggingface_hub import HfApi, create_repo

REPO_NAME = "mapudungun-translator"
REPO_ID = f"{hf_username}/{REPO_NAME}"

print(f"📤 Uploading to HuggingFace: {REPO_ID}")

try:
    # Create repository (if it doesn't exist)
    create_repo(REPO_ID, exist_ok=True)
    print(f"   ✅ Repository created/found: {REPO_ID}")
    
    # Upload all files
    api = HfApi()
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=REPO_ID,
        commit_message="Upload Mapudungun translator model"
    )
    
    print(f"\n✅ Model uploaded successfully!")
    print(f"\n🔗 Your model is available at:")
    print(f"   https://huggingface.co/{REPO_ID}")
    
except Exception as e:
    print(f"❌ Error uploading: {e}")
    print("\n💡 You can manually upload later by running:")
    print(f"   huggingface-cli upload {REPO_ID} {OUTPUT_DIR}")

## 🧪 Step 11: Test Your Translator!

**Now the fun part!**

Let's test your newly trained model with some translations.

**What to expect:**
- Common phrases should translate well
- Complex sentences may have errors
- Results improve with more training data

In [ ]:
def translate(text, source_lang, target_lang, model, tokenizer, max_length=128):
    """
    Translate text from source language to target language
    """
    # Set source language
    tokenizer.src_lang = source_lang
    
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True,
        padding=True
    ).to(model.device)
    
    # Get target language token
    forced_bos_token_id = tokenizer.lang_code_to_id.get(target_lang, tokenizer.bos_token_id)
    
    # Generate translation
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_length=max_length,
            num_beams=5,           # Beam search for better quality
            early_stopping=True
        )
    
    # Decode output
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translation

def translate_es_to_arn(text):
    """Translate Spanish to Mapudungun"""
    return translate(text, SPANISH_CODE, MAPUDUNGUN_CODE, model, tokenizer)

def translate_arn_to_es(text):
    """Translate Mapudungun to Spanish"""
    return translate(text, MAPUDUNGUN_CODE, SPANISH_CODE, model, tokenizer)

print("✅ Translation functions ready!")
print("\n📝 Functions available:")
print("   - translate_es_to_arn(text)  # Spanish → Mapudungun")
print("   - translate_arn_to_es(text)  # Mapudungun → Spanish")

In [ ]:
# Test Spanish → Mapudungun
print("🇪🇸 → 🌿 Spanish to Mapudungun Tests")
print("="*50)

test_spanish = [
    "Hola",
    "Buenos días",
    "¿Cómo estás?",
    "Gracias",
    "Mi nombre es Juan",
    "El agua está fría",
    "Te quiero mucho"
]

for text in test_spanish:
    translation = translate_es_to_arn(text)
    print(f"\n  Spanish:    {text}")
    print(f"  Mapudungun: {translation}")

In [ ]:
# Test Mapudungun → Spanish
print("\n🌿 → 🇪🇸 Mapudungun to Spanish Tests")
print("="*50)

# Use some known Mapudungun words/phrases from our training data
test_mapudungun = [
    "Mari mari",
    "Pewkayal",
    "Chaltu",
]

# Also test with some samples from our test set
if test_data:
    for pair in test_data[:5]:
        test_mapudungun.append(pair['target'])

for text in test_mapudungun[:7]:  # Limit to 7 tests
    translation = translate_arn_to_es(text)
    print(f"\n  Mapudungun: {text}")
    print(f"  Spanish:    {translation}")

In [ ]:
# Interactive translation cell - run this and enter your own text!
print("🎮 Interactive Translator")
print("="*50)
print("\nEnter text to translate (or 'quit' to exit)")
print("Format: 'es: your spanish text' or 'arn: your mapudungun text'")
print("Example: 'es: Buenos días amigo'")
print()

while True:
    user_input = input("Enter text: ").strip()
    
    if user_input.lower() == 'quit':
        print("👋 ¡Pewkayal! (Goodbye!)")
        break
    
    if user_input.startswith('es:'):
        text = user_input[3:].strip()
        result = translate_es_to_arn(text)
        print(f"  → Mapudungun: {result}")
    elif user_input.startswith('arn:'):
        text = user_input[4:].strip()
        result = translate_arn_to_es(text)
        print(f"  → Spanish: {result}")
    else:
        print("  ⚠️ Please start with 'es:' or 'arn:'")
    print()

## 📊 Step 12: Evaluate Model Quality (Optional)

**What this does:**
- Calculates BLEU score (standard translation quality metric)
- Tests on data the model hasn't seen before
- Gives you an objective quality measure

**BLEU score interpretation:**
- 0-10: Almost useless
- 10-20: Some understanding
- 20-30: Good for low-resource languages!
- 30-40: High quality
- 40+: Very high quality (rare for low-resource)

In [ ]:
from sacrebleu.metrics import BLEU
from tqdm import tqdm

def evaluate_model(test_pairs, direction='es_to_arn'):
    """
    Evaluate model on test set using BLEU score
    """
    predictions = []
    references = []
    
    print(f"\n📊 Evaluating {direction}...")
    
    for pair in tqdm(test_pairs, desc="Evaluating"):
        if direction == 'es_to_arn':
            source = pair['source']
            target = pair['target']
            pred = translate_es_to_arn(source)
        else:
            source = pair['target']
            target = pair['source']
            pred = translate_arn_to_es(source)
        
        predictions.append(pred)
        references.append([target])  # BLEU expects list of references
    
    # Calculate BLEU
    bleu = BLEU()
    score = bleu.corpus_score(predictions, references)
    
    return score, predictions, references

# Evaluate both directions
print("🧪 Running evaluation on test set...")
print("   This may take a few minutes.\n")

# Spanish → Mapudungun
bleu_es_arn, pred_es_arn, ref_es_arn = evaluate_model(test_data[:50], 'es_to_arn')
print(f"\n📊 Spanish → Mapudungun BLEU: {bleu_es_arn.score:.2f}")

# Mapudungun → Spanish
bleu_arn_es, pred_arn_es, ref_arn_es = evaluate_model(test_data[:50], 'arn_to_es')
print(f"📊 Mapudungun → Spanish BLEU: {bleu_arn_es.score:.2f}")

# Average
avg_bleu = (bleu_es_arn.score + bleu_arn_es.score) / 2
print(f"\n📊 Average BLEU: {avg_bleu:.2f}")

# Interpretation
print("\n📝 Score interpretation:")
if avg_bleu < 10:
    print("   Model needs more training data")
elif avg_bleu < 20:
    print("   Model has basic understanding - good start!")
elif avg_bleu < 30:
    print("   Good quality for a low-resource language!")
else:
    print("   Excellent quality!")

## 🎉 Congratulations!

You've successfully:

1. ✅ Scraped translation data from Glosbe
2. ✅ Cleaned and prepared the data
3. ✅ Fine-tuned NLLB-200 with LoRA
4. ✅ Saved your model to HuggingFace
5. ✅ Tested translations in both directions

---

### 📥 How to Use Your Model Later

```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

# Load your model from HuggingFace
base_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
model = PeftModel.from_pretrained(base_model, "YOUR_USERNAME/mapudungun-translator")
tokenizer = AutoTokenizer.from_pretrained("YOUR_USERNAME/mapudungun-translator")

# Now use the translate functions from this notebook!
```

---

### 🚀 Next Steps to Improve

1. **More data**: Increase `MAX_WORDS` in Step 4 and re-run
2. **More epochs**: Change `num_train_epochs` to 5-10
3. **Larger model**: Try `nllb-200-1.3B` (needs more GPU memory)
4. **Data augmentation**: Add more sources (books, websites, etc.)

---

### 🙏 Cultural Note

This model is created to help preserve and promote the **Mapudungun** language.
Please use it respectfully and consider contributing back to the Mapuche community.

**Pewkayal!** (Goodbye in Mapudungun)

In [ ]:
# Download model locally (optional)
# Run this if you want to save the model to your computer

from google.colab import files
import shutil

print("📦 Creating downloadable zip file...")

# Create zip of model folder
shutil.make_archive('mapudungun-translator', 'zip', OUTPUT_DIR)

print("✅ Zip file created!")
print("\n📥 Click below to download:")

# Trigger download
files.download('mapudungun-translator.zip')